# 05 · Variables d'analyse et agrégats

**Projet 4 : Analyse des données de vente et retours** · Sprint 3

**Objectif** : calculer, à partir de la table propre, les variables dont les analyses ont besoin (chiffre d'affaires, coûts, marges en valeur, taux de remise effectif, taux de retour), puis préparer des tables agrégées pour les tableaux de bord et les modèles.

| Étape | Ce qu'on calcule | Sert à |
|---|---|---|
| 1. Chargement | La table propre de `04_nettoyage` | |
| 2. Chiffre d'affaires | CA brut, CA net | Q1 |
| 3. Coûts | Coût brut, coût du retour, coût net | Marges |
| 4. Marges en valeur | Marge brute, marge nette | Q1, Q5 |
| 5. Remises | Montant avant remise, taux de remise effectif, indicateurs | Q2, Q4, Q5 |
| 6. Retours | Taux de retour, indicateur | Q3, Q4 |
| 7. Calendrier | Mois, semaine, jour de la semaine, rôle de l'année (référence ou exercice) | Q1, prévision |
| 8. Validation | Cohérence des variables entre elles | |
| 9. Agrégats | Totaux par semaine, boutique, produit, promotion, pays et mois, et par boutique et par année | Tableaux de bord, modèles |
| 10. Enregistrement | | |

**Périmètre** : les ventes complètes de nos 56 boutiques sur trois années (8 catégories, 12 trimestres), voir `00_inventaire`, section 9. L'année 2019 est l'**année de référence**, 2021 est l'**exercice étudié** : c'est sur lui que portent les chiffres de situation. Les deux rôles sont lus dans `src/config.py` (`ANNEE_REFERENCE`, `EXERCICE`), jamais écrits en dur.

**Lit dans** : `data/clean/fait_ventes.parquet`
**Écrit dans** : `data/clean/fait_ventes_variables.parquet` et `data/clean/agregats`

### Les formules, décidées par l'équipe après le diagnostic

| Variable | Formule | Remarque |
|---|---|---|
| `ca_brut` | montant des ventes | Colonne d'origine, inchangée |
| `ca_net` | montant des ventes − montant du retour | |
| `cout_brut` | coût unitaire × quantité vendue | Remplace `cout_total`, calculé de trois façons dans la source |
| `cout_retour` | coût unitaire × quantité retournée | |
| `cout_net` | coût brut − coût du retour | |
| `marge_brute` | CA brut − coût brut | Marge en **valeur** (dollars), pas en taux |
| `marge_nette` | CA net − coût net | |
| `montant_avant_remise` | prix unitaire × quantité vendue | |
| `taux_remise_effectif` | montant de la remise ÷ montant avant remise | Le taux réellement obtenu, différent du taux annoncé |
| `taux_retour` | quantité retournée ÷ quantité vendue | |

**Hypothèse à écrire dans le rapport** : en retirant le coût du retour, on suppose qu'un article retourné est remis en stock et que son coût est récupéré. Les données ne disent rien de l'état des articles retournés. Si un retour est invendable, la marge nette réelle est plus basse que celle calculée ici.

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config -> chemins (CLEAN, AGREGATS)
#   viz    -> charte graphique
from src import config, viz

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 70)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Racine du projet :", config.ROOT)

# Tolérance pour comparer des montants (arrondis au cent dans la source)
TOLERANCE = 0.02

## 1. Chargement

In [ ]:
chemin = config.CLEAN / "fait_ventes.parquet"
if not chemin.exists():
    raise FileNotFoundError(f"{chemin} introuvable : exécuter d'abord 04_nettoyage.")

ventes = pd.read_parquet(chemin)
colonnes_origine = list(ventes.columns)   # pour lister à la fin les variables ajoutées

print(f"Lignes   : {len(ventes):,}".replace(",", " "))
print(f"Colonnes : {ventes.shape[1]}")

# Raccourcis utilisés dans les formules
q, q_remise, q_retour = ventes.quantite_vente, ventes.quantite_remise, ventes.quantite_retour
prix, cout = ventes.prix_unitaire, ventes.cout_unitaire

## 2. Chiffre d'affaires

Le montant des ventes de la source est **brut** : les retours n'y sont pas déduits (diagnostic, section 7). On garde ce montant tel quel sous le nom `ca_brut`, et on calcule le chiffre d'affaires **net** en retirant le montant des retours.

In [ ]:
# CA brut : copie du montant des ventes, sous un nom explicite
ventes["ca_brut"] = ventes.montant_ventes

# CA net : ce qui reste une fois les retours remboursés
ventes["ca_net"] = ventes.montant_ventes - ventes.montant_retour

print(f"CA brut : {ventes.ca_brut.sum():>16,.0f} $".replace(",", " "))
print(f"Retours : {ventes.montant_retour.sum():>16,.0f} $".replace(",", " "),
      f"({ventes.montant_retour.sum() / ventes.ca_brut.sum():.2%} du CA brut)")
print(f"CA net  : {ventes.ca_net.sum():>16,.0f} $".replace(",", " "))

# Un retour est remboursé au prix plein, alors que la vente a pu bénéficier d'une remise.
# Le CA net d'une ligne peut donc, en théorie, être négatif. On le vérifie.
print("Lignes avec un CA net négatif :", int((ventes.ca_net < 0).sum()))

## 3. Coûts

**Pourquoi recalculer ?** Dans la source, la colonne `cout_total` suit trois logiques différentes sur les lignes avec retour (diagnostic, section 7) : tantôt le coût de toutes les unités vendues, tantôt le coût diminué des unités retournées, tantôt diminué du double. Une marge calculée avec cette colonne mélangerait ces logiques.

On calcule donc trois coûts avec **une seule formule chacun, appliquée à toutes les lignes**. La colonne `cout_total` d'origine reste dans la table, sans être utilisée.

In [ ]:
# Coût brut : coût de toutes les unités vendues
ventes["cout_brut"] = cout * q

# Coût du retour : coût des unités retournées (supposées remises en stock)
ventes["cout_retour"] = cout * q_retour

# Coût net : coût des unités vendues et conservées par le client
ventes["cout_net"] = ventes.cout_brut - ventes.cout_retour

# Comparaison avec la colonne d'origine : où et de combien diffère-t-elle ?
comparaison = pd.DataFrame({
    "total": [ventes.cout_total.sum(), ventes.cout_brut.sum(), ventes.cout_retour.sum(), ventes.cout_net.sum()],
}, index=["cout_total (source, logique variable)", "cout_brut", "cout_retour", "cout_net"])
display(comparaison.style.format("{:,.0f} $"))

sans_retour = q_retour == 0
print("Sans retour, cout_brut = cout_total sur toutes les lignes :",
      bool(((ventes.cout_brut - ventes.cout_total).abs() <= TOLERANCE)[sans_retour].all()))
print("Avec retour, lignes où cout_total = cout_brut :",
      int(((ventes.cout_brut - ventes.cout_total).abs() <= TOLERANCE)[~sans_retour].sum()),
      "· lignes où cout_total = cout_net :",
      int(((ventes.cout_net - ventes.cout_total).abs() <= TOLERANCE)[~sans_retour].sum()),
      "· sur", int((~sans_retour).sum()))

## 4. Marges en valeur

L'équipe analyse la marge **en valeur** (en dollars), pas en taux. Deux marges, chacune construite avec un chiffre d'affaires et un coût de même nature :

- **marge brute** = CA brut − coût brut : ce que rapporte la vente avant tout retour ;
- **marge nette** = CA net − coût net : ce qui reste une fois les retours remboursés et les articles repris.

In [ ]:
ventes["marge_brute"] = ventes.ca_brut - ventes.cout_brut
ventes["marge_nette"] = ventes.ca_net - ventes.cout_net

# La différence entre les deux marges est la marge perdue à cause des retours :
#     marge_brute − marge_nette = montant_retour − cout_retour
ventes["marge_perdue_retours"] = ventes.marge_brute - ventes.marge_nette

resume = pd.DataFrame({
    "total": [ventes.marge_brute.sum(), ventes.marge_perdue_retours.sum(), ventes.marge_nette.sum()],
    "par ligne (moyenne)": [ventes.marge_brute.mean(), ventes.marge_perdue_retours.mean(), ventes.marge_nette.mean()],
    "par ligne (médiane)": [ventes.marge_brute.median(), ventes.marge_perdue_retours.median(), ventes.marge_nette.median()],
}, index=["marge brute", "marge perdue à cause des retours", "marge nette"])
display(resume.style.format("{:,.0f} $"))

print("Lignes avec une marge brute négative :", int((ventes.marge_brute < 0).sum()))
print("Lignes avec une marge nette négative :", int((ventes.marge_nette < 0).sum()))

### Et si les articles retournés ne sont pas revendables ?

Notre marge nette suppose que l'article retourné est remis en stock : son coût est récupéré. Si l'article est abîmé ou invendable, ce coût est perdu et le retour coûte plus cher. **Les données ne disent rien de l'état des retours** : on ne peut pas savoir quelle part est revendable.

On peut en revanche **encadrer** la perte entre deux cas extrêmes :

- **tous les retours sont revendables** (notre calcul) : on perd seulement la marge de l'unité, soit montant du retour − coût du retour ;
- **aucun retour n'est revendable** : on rembourse le client et on perd l'article, soit le montant du retour en entier.

In [ ]:
# Marge perdue à cause des retours, dans les deux cas extrêmes
perte_si_revendables = ventes.marge_perdue_retours.sum()   # montant_retour − cout_retour
perte_si_invendables = ventes.montant_retour.sum()         # remboursement entier, article perdu
marge_brute_totale = ventes.marge_brute.sum()

encadrement = pd.DataFrame({
    "marge_perdue": [perte_si_revendables, perte_si_invendables],
    "marge_nette": [marge_brute_totale - perte_si_revendables, marge_brute_totale - perte_si_invendables],
}, index=["tous les retours sont revendables (notre calcul)", "aucun retour n'est revendable"])
encadrement["part_de_la_marge_brute"] = encadrement.marge_perdue / marge_brute_totale
display(encadrement.style.format({"marge_perdue": "{:,.0f} $", "marge_nette": "{:,.0f} $", "part_de_la_marge_brute": "{:.2%}"}))

ecart = perte_si_invendables - perte_si_revendables   # = coût total des unités retournées
print(f"Écart entre les deux cas : {ecart:,.0f} $".replace(",", " "),
      f"(le coût des unités retournées), soit {ecart / marge_brute_totale:.2%} de la marge brute.")
print("La vraie valeur se situe entre les deux. L'incertitude existe, mais elle ne change pas les ordres de grandeur.")

## 5. Remises

**Constat du diagnostic (section 8)** : le taux annoncé d'une promotion (5 à 20 %) est rarement le taux obtenu, parce que le plus souvent seule une partie des unités d'une ligne est remisée. Le mécanisme est inconnu : la table des promotions ne décrit aucune règle.

**Décision** : on ne corrige pas ce mécanisme, les montants de remise sont gardés. Seule exception, traitée dans `04_nettoyage` (règle N6) : sur les 2 331 lignes où la remise portait sur plus d'unités que la vente, la quantité remisée a été ramenée à la quantité vendue, **sans modifier les montants** (indicateur `remise_corrigee`). Sur ces lignes, le taux de remise effectif dépasse le taux annoncé (jusqu'à 40 % pour 20 %). Partout ailleurs il lui est inférieur ou égal, ce que la validation de la section 8 vérifie. Dans une analyse des remises, l'indicateur permet d'inclure ou d'exclure ces lignes. On calcule le **taux de remise effectif** de chaque ligne, c'est-à-dire le vrai taux de remise de ce produit, dans cette boutique, ce jour-là. C'est cette variable, continue, qui servira à Q2, Q4 et Q5.

In [ ]:
# Montant avant remise : ce que la ligne aurait rapporté au prix plein
ventes["montant_avant_remise"] = prix * q

# Taux de remise effectif : part du montant avant remise qui a été abandonnée au client
ventes["taux_remise_effectif"] = ventes.montant_remise / ventes.montant_avant_remise

# Part des unités de la ligne qui ont été remisées
ventes["part_unites_remisees"] = q_remise / q

# Indicateurs (vrai / faux)
#   a_promotion : la ligne est rattachée à une promotion (id_promotion 1 = "No Discount")
#   a_remise    : la ligne porte réellement une remise
# Les deux ne coïncident pas : une ligne peut être rattachée à une promotion sans aucune unité remisée.
ventes["a_promotion"] = ventes.id_promotion != 1
ventes["a_remise"] = ventes.montant_remise > 0

display(pd.crosstab(ventes.a_promotion.map({True: "rattachée à une promotion", False: "sans promotion"}),
                    ventes.a_remise.map({True: "remise > 0", False: "remise = 0"})))

# Taux annoncé contre taux effectif
comparaison_taux = (ventes[ventes.a_promotion].groupby("taux_remise_annonce").taux_remise_effectif
                    .agg(lignes="size", moyenne="mean", mediane="median", maximum="max"))
display(comparaison_taux.style.format({"lignes": "{:,.0f}", "moyenne": "{:.2%}", "mediane": "{:.2%}", "maximum": "{:.2%}"}))

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : distribution du taux de remise effectif, selon le taux annoncé
# ---------------------------------------------------------------------------
taux_annonces = sorted(ventes.loc[ventes.a_promotion, "taux_remise_annonce"].unique())
fig, ax = plt.subplots(figsize=(9, 4.5))
boites = ax.boxplot([ventes.loc[ventes.a_promotion & (ventes.taux_remise_annonce == t), "taux_remise_effectif"] for t in taux_annonces],
                    patch_artist=True, widths=0.55, showfliers=False,
                    medianprops=dict(color=viz.ACCENT, linewidth=2))
for boite in boites["boxes"]:
    boite.set(facecolor=viz.CLAIR, edgecolor=viz.PRINCIPALE)
# Repère rouge : le taux annoncé, que le taux effectif devrait atteindre si toute la ligne était remisée
for i, t in enumerate(taux_annonces, start=1):
    ax.hlines(t, i - 0.35, i + 0.35, color=viz.ALERTE, linewidth=2, linestyles="--")
ax.set_xticks(range(1, len(taux_annonces) + 1), [f"{t:.0%}" for t in taux_annonces])
ax.set_xlabel("Taux de remise annoncé par la promotion")
ax.set_ylabel("Taux de remise effectif de la ligne")
# Graduations tous les 5 points : sans cela, matplotlib place des graduations à 2,5 % ou 7,5 %,
# que l'affichage sans décimale arrondirait à 2 % et 8 %.
ax.set_yticks(np.arange(0, max(taux_annonces) + 0.051, 0.05))
viz.format_pourcent(ax, axe="y")
viz.titre(ax, "Le taux de remise obtenu reste loin du taux annoncé",
          "Boîtes : taux effectif des lignes en promotion · trait orange : médiane · pointillé rouge : taux annoncé")
print("Figure enregistrée :", viz.enregistrer(fig, "05_taux_remise_effectif"))
plt.show()

## 6. Retours

Le taux de retour d'une ligne est la part des unités vendues qui ont été retournées. On ajoute un indicateur `a_retour`, qui sera la variable à expliquer pour le modèle de risque de retour.

In [ ]:
ventes["a_retour"] = q_retour > 0
ventes["taux_retour"] = q_retour / q

print(f"Lignes avec au moins un retour : {ventes.a_retour.mean():.1%}")
print(f"Taux de retour global en quantité : {q_retour.sum() / q.sum():.2%}")
print(f"Taux de retour global en valeur   : {ventes.montant_retour.sum() / ventes.ca_brut.sum():.2%}")
print()
# ATTENTION à la lecture. Deux chiffres très différents :
#   - part des LIGNES avec un retour (élevée) : une ligne cumule plusieurs unités, il suffit d'une unité retournée ;
#   - part des UNITÉS retournées (faible) : c'est le vrai taux de retour.
# Pour un taux global, on divise toujours des SOMMES (retours ÷ ventes), jamais on ne fait
# la moyenne des taux par ligne : une ligne de 6 unités pèserait autant qu'une ligne de 2 000.
print(f"À ne pas utiliser comme taux global : moyenne des taux par ligne = {ventes.taux_retour.mean():.2%}")

## 7. Calendrier

Variables de temps utiles aux analyses de saisonnalité et à la prévision. La semaine est repérée par son **lundi**, ce qui évite les ambiguïtés de numérotation en début et fin d'année.

Deux indicateurs donnent le **rôle de l'année** dans l'étude : `annee_de_reference` (2019, le point de comparaison) et `exercice_etudie` (2021, la situation à expliquer). Ils évitent d'écrire une année en dur dans les notebooks d'analyse.

In [ ]:
ventes["annee_mois"] = ventes.date.dt.to_period("M").astype(str)                       # ex. "2020-03"
ventes["semaine_debut"] = ventes.date - pd.to_timedelta(ventes.date.dt.dayofweek, unit="D")   # lundi de la semaine
ventes["jour_semaine"] = ventes.date.dt.dayofweek + 1                                  # 1 = lundi ... 7 = dimanche
NOMS_JOURS = {1: "lundi", 2: "mardi", 3: "mercredi", 4: "jeudi", 5: "vendredi", 6: "samedi", 7: "dimanche"}
ventes["nom_jour"] = ventes.jour_semaine.map(NOMS_JOURS)
ventes["fin_de_semaine"] = ventes.jour_semaine >= 6

# Rôle de l'année dans l'étude (lu dans config, pour ne jamais écrire une année en dur)
ventes["annee_de_reference"] = ventes.annee == config.ANNEE_REFERENCE
ventes["exercice_etudie"] = ventes.annee == config.EXERCICE
print("Année de référence :", config.ANNEE_REFERENCE, "· exercice étudié :", config.EXERCICE)
print(ventes.groupby("annee").agg(lignes=("id_vente", "size"), annee_de_reference=("annee_de_reference", "first"),
                                  exercice_etudie=("exercice_etudie", "first")).to_string(), "\n")

display(ventes[["date", "annee", "trimestre", "mois", "annee_mois", "semaine_debut", "jour_semaine", "nom_jour", "fin_de_semaine"]].head(3))

## 8. Validation des variables

On vérifie que les variables sont cohérentes entre elles et avec la source. Si un contrôle échoue, rien n'est écrit.

In [ ]:
def proche(a, b):
    """Vrai si deux séries sont égales à la tolérance près, sur toutes les lignes."""
    return bool(((a - b).abs() <= TOLERANCE).all())


controles = {
    "ca_brut = montant_ventes": proche(ventes.ca_brut, ventes.montant_ventes),
    "ca_net = ca_brut − montant_retour": proche(ventes.ca_net, ventes.ca_brut - ventes.montant_retour),
    "cout_net = cout_unitaire × (quantité vendue − quantité retournée)": proche(ventes.cout_net, cout * (q - q_retour)),
    "cout_brut = cout_retour + cout_net": proche(ventes.cout_brut, ventes.cout_retour + ventes.cout_net),
    "ca_brut = montant_avant_remise − montant_remise (règle de l'énoncé)": proche(ventes.ca_brut, ventes.montant_avant_remise - ventes.montant_remise),
    "marge perdue = montant_retour − cout_retour": proche(ventes.marge_perdue_retours, ventes.montant_retour - ventes.cout_retour),
    "taux_remise_effectif entre 0 et 1": bool(ventes.taux_remise_effectif.between(0, 1).all()),
    # Hors lignes de la règle N6 (remise_corrigee), où la remise conservée dépasse le taux annoncé
    "taux_remise_effectif ≤ taux annoncé (hors lignes remise_corrigee)": bool(
        (ventes.taux_remise_effectif <= ventes.taux_remise_annonce + 1e-6)[~ventes.remise_corrigee].all()),
    "part_unites_remisees entre 0 et 1": bool(ventes.part_unites_remisees.between(0, 1).all()),
    "taux_retour entre 0 et 1": bool(ventes.taux_retour.between(0, 1).all()),
    "remise nulle sans promotion": bool((ventes.loc[~ventes.a_promotion, "montant_remise"] == 0).all()),
    "aucun vide dans les nouvelles variables": bool(ventes[[c for c in ventes.columns if c not in colonnes_origine]].notna().all().all()),
}
for nom, ok in controles.items():
    print(f"[{'réussi' if ok else 'ÉCHOUÉ'}] {nom}")
assert all(controles.values()), "Validation échouée : rien n'est écrit."

nouvelles = [c for c in ventes.columns if c not in colonnes_origine]
print(f"\n{len(nouvelles)} variables ajoutées :", nouvelles)

## 9. Agrégats

Tables de totaux prêtes à l'emploi pour Excel, Power BI et les modèles. Une seule fonction construit tous les agrégats, pour que les indicateurs soient calculés **partout de la même façon**.

**Règle importante** : dans un agrégat, les taux sont recalculés à partir des **sommes** (somme des remises ÷ somme des montants avant remise). On ne fait jamais la moyenne des taux des lignes.

In [ ]:
def agreger(df, cles):
    """Totaux et indicateurs par groupe.

    df   : table des ventes avec les variables de ce notebook
    cles : liste des colonnes de regroupement (ex. ["semaine_debut"], ["pays", "annee_mois"])
    Renvoie une table avec une ligne par groupe.
    """
    g = df.groupby(cles, observed=True)
    resultat = g.agg(
        nb_lignes=("id_vente", "size"),
        quantite_vente=("quantite_vente", "sum"),
        quantite_retour=("quantite_retour", "sum"),
        quantite_remise=("quantite_remise", "sum"),
        montant_avant_remise=("montant_avant_remise", "sum"),
        montant_remise=("montant_remise", "sum"),
        montant_retour=("montant_retour", "sum"),
        ca_brut=("ca_brut", "sum"),
        ca_net=("ca_net", "sum"),
        cout_brut=("cout_brut", "sum"),
        cout_retour=("cout_retour", "sum"),
        cout_net=("cout_net", "sum"),
        marge_brute=("marge_brute", "sum"),
        marge_nette=("marge_nette", "sum"),
        nb_lignes_avec_retour=("a_retour", "sum"),
        nb_lignes_avec_remise=("a_remise", "sum"),
    ).reset_index()
    # Taux recalculés à partir des sommes du groupe
    resultat["taux_remise_effectif"] = resultat.montant_remise / resultat.montant_avant_remise
    resultat["taux_retour"] = resultat.quantite_retour / resultat.quantite_vente
    resultat["part_lignes_avec_retour"] = resultat.nb_lignes_avec_retour / resultat.nb_lignes
    return resultat


agregats = {
    # Série hebdomadaire : base de la prévision des ventes
    "par_semaine": agreger(ventes, ["semaine_debut"]),
    # Par boutique : performance des points de vente (Q1)
    "par_boutique": agreger(ventes, ["id_boutique", "nom_boutique", "type_boutique", "pays", "code_pays"]),
    # Par produit : performance et retours par produit (Q1, Q3)
    "par_produit": agreger(ventes, ["id_produit", "nom_produit", "categorie", "sous_categorie", "marque", "classe"]),
    # Par promotion : effet des promotions (Q2, Q4)
    "par_promotion": agreger(ventes, ["id_promotion", "nom_promotion", "taux_remise_annonce"]),
    # Par pays et par mois : carte et évolution dans les tableaux de bord
    "par_pays_mois": agreger(ventes, ["pays", "code_pays", "annee_mois"]),
}

# ---------------------------------------------------------------------------
# Par boutique et par année : la base de toute comparaison entre magasins
# ---------------------------------------------------------------------------
# Six magasins ouvrent pendant la période (diagnostic, section 9). Pour comparer des magasins,
# il faut savoir combien de mois chacun a réellement été ouvert dans l'année :
#   mois_actifs    : nombre de mois avec au moins une vente APRÈS l'ouverture (hors lignes avant_ouverture)
#   annee_complete : vrai si la boutique était déjà ouverte le 1er janvier de l'année
par_boutique_annee = agreger(ventes, ["id_boutique", "nom_boutique", "type_boutique", "pays", "annee"])
mois_actifs = (ventes[~ventes.avant_ouverture].groupby(["id_boutique", "annee"]).annee_mois.nunique()
               .rename("mois_actifs").reset_index())
ouverture = ventes.groupby("id_boutique").date_ouverture.first().rename("date_ouverture").reset_index()
par_boutique_annee = (par_boutique_annee.merge(mois_actifs, on=["id_boutique", "annee"], how="left")
                      .merge(ouverture, on="id_boutique", how="left"))
par_boutique_annee["mois_actifs"] = par_boutique_annee.mois_actifs.fillna(0).astype(int)
par_boutique_annee["annee_complete"] = par_boutique_annee.date_ouverture < pd.to_datetime(par_boutique_annee.annee.astype(str) + "-01-02")
agregats["par_boutique_annee"] = par_boutique_annee

for nom, table in agregats.items():
    # Contrôle : la somme de chaque agrégat redonne le total général
    assert abs(table.ca_brut.sum() - ventes.ca_brut.sum()) < 1, f"{nom} : le CA brut ne se recoupe pas"
    assert table.nb_lignes.sum() == len(ventes), f"{nom} : le nombre de lignes ne se recoupe pas"
    print(f"{nom:<15} {len(table):>5} lignes · total CA brut recoupé")

print("\nMagasins selon leur activité dans l'année (type Store) :")
magasins_annee = par_boutique_annee[par_boutique_annee.type_boutique == "Store"]
display(magasins_annee.groupby("annee").agg(
    magasins_avec_ventes=("id_boutique", "nunique"),
    ouverts_toute_l_annee=("annee_complete", "sum"),
    ca_brut_total=("ca_brut", "sum"),
    ca_brut_des_magasins_ouverts_toute_l_annee=("ca_brut", lambda s: s[magasins_annee.loc[s.index, "annee_complete"]].sum()),
).style.format({"ca_brut_total": "{:,.0f}", "ca_brut_des_magasins_ouverts_toute_l_annee": "{:,.0f}"}))
print("Magasins qui ne sont pas ouverts toute l'année :")
display(magasins_annee.loc[~magasins_annee.annee_complete, ["nom_boutique", "annee", "date_ouverture", "mois_actifs", "nb_lignes", "ca_brut"]]
        .sort_values(["date_ouverture", "annee"]).style.format({"ca_brut": "{:,.0f}", "date_ouverture": "{:%Y-%m-%d}"}).hide(axis="index"))

print("\nAgrégat par promotion :")
display(agregats["par_promotion"][["id_promotion", "nom_promotion", "taux_remise_annonce", "nb_lignes", "ca_brut",
                                   "marge_nette", "taux_remise_effectif", "taux_retour"]]
        .style.format({"taux_remise_annonce": "{:.0%}", "nb_lignes": "{:,.0f}", "ca_brut": "{:,.0f}",
                       "marge_nette": "{:,.0f}", "taux_remise_effectif": "{:.2%}", "taux_retour": "{:.2%}"}))

**À lire dans le tableau des magasins** : le nombre de magasins ouverts toute l'année passe de 48 en 2019 à 51 en 2020 et 53 en 2021. Les magasins ouverts en cours d'année pèsent peu dans le total, mais ils faussent toute moyenne par magasin et tout décompte de magasins bénéficiaires si on les traite comme des magasins ouverts douze mois. Les notebooks d'analyse filtrent donc sur `annee_complete` pour comparer des magasins, ou ramènent les montants au mois d'activité (`mois_actifs`).

**Attention pour la série hebdomadaire** : avec les ventes complètes, la série couvre les trois années sans interruption. Seules la toute première et la toute dernière semaine sont incomplètes (l'année ne commence pas un lundi et ne finit pas un dimanche) : il faudra les écarter ou les traiter à part dans la prévision.

## 10. Enregistrement

In [ ]:
# Table des ventes avec toutes les variables : c'est elle que lisent les notebooks d'analyse et de modélisation.
# On l'écrit sous un nouveau nom, sans écraser fait_ventes.parquet : chaque étape lit le fichier de
# l'étape précédente et écrit le sien, ce qui permet de relancer 05 sans relancer 04.
chemin_variables = config.CLEAN / "fait_ventes_variables.parquet"
ventes.to_parquet(chemin_variables, index=False)
print("Ventes avec variables :", chemin_variables, f"({len(ventes):,} lignes × {ventes.shape[1]} colonnes)".replace(",", " "))

config.AGREGATS.mkdir(parents=True, exist_ok=True)
for nom, table in agregats.items():
    table.to_parquet(config.AGREGATS / f"{nom}.parquet", index=False)
    print("Agrégat :", config.AGREGATS / f"{nom}.parquet")

**Suite** : les notebooks d'analyse (`06` à `09`) et de modélisation (`10` à `13`) lisent `data/clean/fait_ventes_variables.parquet` et `data/clean/agregats`.

**Limites à rappeler dans chaque analyse** (diagnostic, section 11) : le jeu de données est généré, une ligne est un cumul journalier et non un ticket, les prix catalogue ne varient jamais, et le mécanisme des promotions est inconnu.